In [1]:
import os

folders = ["data", "outputs", "visualization"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created")

Project folders created


In [7]:
import requests
import json

repo = "pytorch/pytorch"
url = f"https://api.github.com/repos/{repo}/issues"

params = {
    "state": "all",
    "per_page": 50
}

response = requests.get(url, params=params)

issues = []

for issue in response.json():
    
    issues.append({
        "id": issue["id"],
        "title": issue["title"],
        "body": issue["body"],
        "user": issue["user"]["login"],
        "created_at": issue["created_at"]
    })

with open("data/issues.json", "w") as f:
    json.dump(issues, f, indent=2)

print("Downloaded", len(issues), "issues")

Downloaded 50 issues


In [8]:
import json

with open("data/issues.json") as f:
    issues = json.load(f)

print(len(issues))
print(issues[0])

50
{'id': 4023011364, 'title': '[Inductor][MPS] Fix half-precision type mismatches in Metal shader codegen', 'body': 'Metal Shading Language rejects implicit float-to-bfloat conversions, so\nbare float literals like `0.0` in generated shaders cause compilation\nfailures when the target variable is `bfloat` (or `half`). Three codegen\nmethods were affected:\n\n- `constant()` ignored its `dtype` parameter and returned raw literals.\n- `masked()` assigned a bare literal in the else-branch (`} else tmp = 0.0;`).\n- `where()` passed a bare literal through the ternary without casting.\n\nAll three now emit `static_cast<bfloat>(...)` / `static_cast<half>(...)`\nwhere needed. Tests added for half-precision constants, reductions, and\nconditionals.\n\n\ncc @voznesenskym @penguinwu @EikanWang @jgong5 @Guobing-Chen @XiaobingSuper @zhuhaozhe @blzheng @wenzhe-nrv @jiayisunx @ipiszy @kadeng @muchulee8 @amjames @chauhang @aakhundov @coconutruben @jataylo', 'user': 'mergennachin', 'created_at': '2026-

In [1]:
import requests

url = "http://localhost:11434/api/generate"

payload = {
    "model": "llama3",
    "prompt": "Say hello in one sentence",
    "stream": False
}

response = requests.post(url, json=payload)

print(response.json()["response"])

Hello there!


In [2]:
import json

with open("data/issues.json") as f:
    issues = json.load(f)

print(len(issues))
print(issues[0]["title"])

50
[Inductor][MPS] Fix half-precision type mismatches in Metal shader codegen


In [3]:
def extract_knowledge(issue_text):

    prompt = f"""
You are an information extraction system.

Extract structured knowledge from the following text.

Return JSON with:

entities: list of entities
claims: relationships between entities
evidence: exact supporting text

Text:
{issue_text}

Return ONLY JSON.
"""

    url = "http://localhost:11434/api/generate"

    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)

    return response.json()["response"]

In [4]:
sample_issue = issues[0]["title"] + " " + str(issues[0]["body"])

result = extract_knowledge(sample_issue)

print(result)

Here is the extracted knowledge in JSON format:

```
{
  "entities": [
    {"name": "Inductor", "type": "organization"},
    {"name": "MPS", "type": "technology"},
    {"name": "Metal", "type": "technology"},
    {"name": "Metal Shading Language", "type": "language"},
    {"name": "bfloat", "type": "datatype"},
    {"name": "half", "type": "datatype"},
    {"name": "voznesenskym", "type": "person"},
    {"name": "penguinwu", "type": "person"},
    {"name": "EikanWang", "type": "person"},
    {"name": "jgong5", "type": "person"},
    {"name": "Guobing-Chen", "type": "person"},
    {"name": "XiaobingSuper", "type": "person"},
    {"name": "zhuhaozhe", "type": "person"},
    {"name": "blzheng", "type": "person"},
    {"name": "wenzhe-nrv", "type": "person"},
    {"name": "jiayisunx", "type": "person"},
    {"name": "ipiszy", "type": "person"},
    {"name": "kadeng", "type": "person"},
    {"name": "muchulee8", "type": "person"},
    {"name": "amjames", "type": "person"},
    {"name": "cha

In [5]:
import networkx as nx
import json

In [6]:
G = nx.MultiDiGraph()

In [7]:
def add_entity(entity):
    
    if entity not in G:
        G.add_node(entity, type="entity")


def add_claim(subject, relation, obj, evidence):

    add_entity(subject)
    add_entity(obj)

    G.add_edge(
        subject,
        obj,
        relation=relation,
        evidence=evidence
    )

In [24]:
import json

def process_extraction(output_text):

    try:
        data = json.loads(output_text)
        
        entities = data.get("entities", [])
        claims = data.get("claims", [])
        evidence = data.get("evidence", "Extracted from GitHub issue discussion")

        for entity in entities:
            add_entity(entity)

        for claim in claims:
            
            parts = claim.split()

            if len(parts) >= 3:
                subject = parts[0]
                relation = parts[1]
                obj = parts[2]

                add_claim(subject, relation, obj, evidence)

    except:
        pass

In [9]:
from tqdm import tqdm

for issue in tqdm(issues[:10]):
    
    text = issue["title"] + " " + str(issue["body"])

    output = extract_knowledge(text)

    process_extraction(output)

100%|███████████████████████████████████████████| 10/10 [03:22<00:00, 20.23s/it]


In [10]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 0
Edges: 0


In [11]:
sample_issue = issues[0]["title"] + " " + str(issues[0]["body"])

output = extract_knowledge(sample_issue)

print(output)

Here is the extracted structured knowledge in JSON format:

```
{
  "entities": [
    {"name": "Inductor", "type": "Organization"},
    {"name": "MPS", "type": "Organization"},
    {"name": "voznesenskym", "type": "Person"},
    {"name": "penguinwu", "type": "Person"},
    {"name": "EikanWang", "type": "Person"},
    {"name": "jgong5", "type": "Person"},
    {"name": "Guobing-Chen", "type": "Person"},
    {"name": "XiaobingSuper", "type": "Person"},
    {"name": "zhuhaozhe", "type": "Person"},
    {"name": "blzheng", "type": "Person"},
    {"name": "wenzhe-nrv", "type": "Person"},
    {"name": "jiayisunx", "type": "Person"},
    {"name": "ipiszy", "type": "Person"},
    {"name": "kadeng", "type": "Person"},
    {"name": "muchulee8", "type": "Person"},
    {"name": "amjames", "type": "Person"},
    {"name": "chauhang", "type": "Person"},
    {"name": "aakhundov", "type": "Person"},
    {"name": "coconutruben", "type": "Person"},
    {"name": "jataylo", "type": "Person"}
  ],
  "claims":

In [12]:
import json
import re

def process_extraction(output_text):

    try:
        json_text = re.search(r'\{.*\}', output_text, re.DOTALL).group()
        data = json.loads(json_text)

        entities = data.get("entities", [])
        claims = data.get("claims", [])

        for entity in entities:
            name = entity.get("name")
            if name:
                add_entity(name)

        for claim in claims:

            subject = claim.get("subject")
            relation = claim.get("predicate")
            obj = claim.get("object")
            evidence = claim.get("evidence", "")

            if subject and obj:
                add_claim(subject, relation, obj, evidence)

    except Exception as e:
        print("Parsing error:", e)

In [13]:
G = nx.MultiDiGraph()

In [14]:
from tqdm import tqdm

for issue in tqdm(issues[:10]):

    text = issue["title"] + " " + str(issue["body"])

    output = extract_knowledge(text)

    process_extraction(output)

100%|███████████████████████████████████████████| 10/10 [04:20<00:00, 26.05s/it]


In [15]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 118
Edges: 69


In [16]:
from pyvis.network import Network

net = Network(height="750px", width="100%", directed=True)

for node in G.nodes():
    net.add_node(node, label=node)

for edge in G.edges(data=True):

    source = edge[0]
    target = edge[1]
    relation = edge[2].get("relation", "")

    net.add_edge(source, target, label=relation)

net.show("visualization/memory_graph.html")

visualization/memory_graph.html


AttributeError: 'NoneType' object has no attribute 'render'

In [17]:
from pyvis.network import Network

net = Network(height="750px", width="100%", directed=True, notebook=False)

for node in G.nodes():
    net.add_node(node, label=node)

for edge in G.edges(data=True):

    source = edge[0]
    target = edge[1]
    relation = edge[2].get("relation", "")

    net.add_edge(source, target, label=relation)

net.write_html("visualization/memory_graph.html")

In [18]:
import webbrowser
webbrowser.open("visualization/memory_graph.html")

True

In [19]:
claims_data = []

for u, v, data in G.edges(data=True):
    
    relation = data.get("relation", "")
    evidence = data.get("evidence", "")

    claim_text = f"{u} {relation} {v}"

    claims_data.append({
        "claim": claim_text,
        "evidence": evidence
    })

print(len(claims_data))

69


In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
import numpy as np

claim_texts = [c["claim"] for c in claims_data]

claim_embeddings = model.encode(claim_texts)

print("Embeddings created:", len(claim_embeddings))

Embeddings created: 69


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, top_k=5):

    q_embedding = model.encode([question])

    sims = cosine_similarity(q_embedding, claim_embeddings)[0]

    top_indices = sims.argsort()[-top_k:][::-1]

    results = []

    for i in top_indices:

        results.append({
            "claim": claims_data[i]["claim"],
            "evidence": claims_data[i]["evidence"]
        })

    return results

In [25]:
results = retrieve("Which functions ignore dtype?")

for r in results:
    print("CLAIM:", r["claim"])
    print("EVIDENCE:", r["evidence"])
    print()

CLAIM: constant() ignores dtype parameter
EVIDENCE: constant() ignored its dtype parameter and returned raw literals.

CLAIM: Triton uses ROCm
EVIDENCE: 

CLAIM: generated_serialization_types.h contains auto-generated C++ types with nlohmann::json support
EVIDENCE: 

CLAIM: C++ schema types registered as opaque pybind11 types with no field access from Python
EVIDENCE: 

CLAIM: torch._C._export.deserialize_exported_program() already performs C++ JSON parsing and schema construction
EVIDENCE: 



In [26]:
import json

graph_data = nx.node_link_data(G)

with open("outputs/memory_graph.json", "w") as f:
    json.dump(graph_data, f, indent=2)

print("Memory graph saved")

Memory graph saved


In [27]:
questions = [
    "Which functions ignore dtype?",
    "What issues mention CUDA?",
    "Which systems use Triton?",
    "What serialization components exist?"
]

context_packs = {}

for q in questions:
    context_packs[q] = retrieve(q)

with open("outputs/context_packs.json", "w") as f:
    json.dump(context_packs, f, indent=2)

print("Context packs saved")

Context packs saved
